In [10]:
import ast
from dataclasses import dataclass
from typing import Any

import sympy as sp
from sympy.printing.latex import LatexPrinter


@dataclass(frozen=True)
class Meta:
    kind: str
    children: tuple[Any, ...]


def is_matrix_value(v):
    return isinstance(v, sp.MatrixBase) or (
        isinstance(v, (list, tuple))
        and len(v) > 0
        and all(isinstance(row, (list, tuple)) for row in v)
    )


def to_matrix(v):
    if isinstance(v, sp.MatrixBase):
        return sp.ImmutableDenseMatrix(v)

    return sp.ImmutableDenseMatrix(v)


def shape_of(v):
    return to_matrix(v).shape


def exact_value(v):
    if is_matrix_value(v):
        return to_matrix(v)

    if isinstance(v, float):
        return sp.Rational(str(v))

    return sp.sympify(v)


class OrderedSympyBuilder(ast.NodeVisitor):
    def __init__(self, source: str, values=None):
        self.source = source
        self.values = values or {}
        self.meta = {}

        self.matrix_shapes = {
            name: shape_of(value)
            for name, value in self.values.items()
            if is_matrix_value(value)
        }

    def remember(self, expr, meta):
        self.meta[id(expr)] = meta
        return expr

    def build(self):
        tree = ast.parse(self.source, mode="eval")
        return self.visit(tree.body)

    def visit_Name(self, node):
        name = node.id

        if name in self.matrix_shapes:
            rows, cols = self.matrix_shapes[name]
            return sp.MatrixSymbol(name, rows, cols)

        if name == "pi":
            return sp.pi

        return sp.Symbol(name)

    def visit_Constant(self, node):
        if isinstance(node.value, int):
            return sp.Integer(node.value)

        if isinstance(node.value, float):
            return sp.Float(str(node.value))

        raise ValueError(f"Unsupported constant: {node.value!r}")

    def visit_BinOp(self, node):
        left = self.visit(node.left)
        right = self.visit(node.right)

        left_is_matrix = getattr(left, "is_Matrix", False)
        right_is_matrix = getattr(right, "is_Matrix", False)

        if isinstance(node.op, ast.Mult):
            if left_is_matrix or right_is_matrix:
                expr = sp.MatMul(left, right, evaluate=False)
            else:
                expr = sp.Mul(left, right, evaluate=False)

            return self.remember(expr, Meta("mul", (left, right)))

        if isinstance(node.op, ast.Add):
            if left_is_matrix or right_is_matrix:
                expr = sp.MatAdd(left, right, evaluate=False)
            else:
                expr = sp.Add(left, right, evaluate=False)

            return self.remember(expr, Meta("add", (left, right)))

        if isinstance(node.op, ast.Sub):
            if left_is_matrix or right_is_matrix:
                expr = sp.MatAdd(left, -right, evaluate=False)
            else:
                expr = sp.Add(
                    left,
                    sp.Mul(-1, right, evaluate=False),
                    evaluate=False,
                )

            return self.remember(expr, Meta("sub", (left, right)))

        raise ValueError(f"Unsupported operator: {ast.dump(node.op)}")

    def generic_visit(self, node):
        raise ValueError(f"Unsupported syntax: {ast.dump(node)}")


class OrderedLatexPrinter(LatexPrinter):
    def __init__(self, meta=None, substitutions=None, **settings):
        self.meta = meta or {}
        self.substitutions = substitutions or {}
        super().__init__(settings)

    def m(self, expr):
        return self.meta.get(id(expr))

    def _print_Symbol(self, expr, **kwargs):
        if expr in self.substitutions:
            return self._print(self.substitutions[expr])

        return super()._print_Symbol(expr, **kwargs)

    def _print_MatrixSymbol(self, expr):
        if expr in self.substitutions:
            return self._print(self.substitutions[expr])

        return super()._print_MatrixSymbol(expr)

    def _print_MatMul(self, expr):
        meta = self.m(expr)

        if meta and meta.kind == "mul":
            left, right = meta.children
            return self._print(left) + r"\," + self._print(right)

        return super()._print_MatMul(expr)

    def _print_Mul(self, expr):
        meta = self.m(expr)

        if meta and meta.kind == "mul":
            left, right = meta.children
            return self._print(left) + self._print(right)

        return super()._print_Mul(expr)

    def _print_MatAdd(self, expr):
        meta = self.m(expr)

        if meta and meta.kind in {"add", "sub"}:
            left, right = meta.children
            op = " + " if meta.kind == "add" else " - "
            return self._print(left) + op + self._print(right)

        return super()._print_MatAdd(expr)


class CalcExpression:
    def __init__(self, lhs, expression, **values):
        self.lhs = lhs
        self.source = expression

        builder = OrderedSympyBuilder(expression, values=values)
        self.sympy = builder.build()
        self.meta = builder.meta

        self.subs = {}

        for name, value in values.items():
            if is_matrix_value(value):
                rows, cols = shape_of(value)
                key = sp.MatrixSymbol(name, rows, cols)
            else:
                key = sp.Symbol(name)

            self.subs[key] = exact_value(value)

    def latex(self, substituted=False):
        printer = OrderedLatexPrinter(
            self.meta,
            substitutions=self.subs if substituted else {},
            mul_symbol="",
        )

        return printer.doprint(self.sympy)

    def result(self):
        return self.sympy.subs(self.subs).doit()

    def report(self):
        return "\n".join(
            [
                rf"{self.lhs} = {self.latex(False)}",
                "=",
                rf"{self.lhs} = {self.latex(True)}",
                "=",
                rf"{self.lhs} = {sp.latex(self.result())}",
            ]
        )
    def report_latex(self):
        return rf"""
    \begin{{aligned}}
    {self.lhs} &= {self.latex(False)} \\[6pt]
    &= {self.latex(True)} \\[6pt]
    &= {(self.result())}
    \end{{aligned}}
    """

In [13]:
A = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]

B = [
    [9, 8, 7],
    [6, 5, 4],
    [3, 2, 1],
]
s = 2 
C = CalcExpression(
    "C",
    "A*B*s",
    A=A,
    B=B,
    s=s
)

print(C.report())

C = A\,B\,s
=
C = \left[\begin{matrix}1 & 2 & 3\\4 & 5 & 6\\7 & 8 & 9\end{matrix}\right]\,\left[\begin{matrix}9 & 8 & 7\\6 & 5 & 4\\3 & 2 & 1\end{matrix}\right]\,2
=
C = \left[\begin{matrix}60 & 48 & 36\\168 & 138 & 108\\276 & 228 & 180\end{matrix}\right]


In [14]:
from IPython.display import display, Math

display(Math((C.report_latex())))

<IPython.core.display.Math object>

In [3]:
Q = CalcExpression(
    "Q",
    "Integral(sin(t), (t, 0, pi)) + Subs(Derivative(t**2 + 3*t, t), t, x)",
    x=5,
)

print(Q.report())

ValueError: Unsupported syntax: Call(func=Name(id='Integral', ctx=Load()), args=[Call(func=Name(id='sin', ctx=Load()), args=[Name(id='t', ctx=Load())]), Tuple(elts=[Name(id='t', ctx=Load()), Constant(value=0), Name(id='pi', ctx=Load())], ctx=Load())])

In [4]:
display(Math((Q.report_latex())))

NameError: name 'Math' is not defined

In [37]:
import sympy as sp
from IPython.display import display, Math

# ----------------------------------------------------------
# Structural engineering calculation helpers
# ----------------------------------------------------------

def area_of_bar(d):
    """
    Area of one circular reinforcement bar.
    A_s = pi*d^2/4
    """
    return CalcExpression(
        "A_s",
        "pi*d**2/4",
        d=d
    )


def total_area_of_bars(n, d):
    """
    Total area of n reinforcement bars.
    A_s = n*pi*d^2/4
    """
    return CalcExpression(
        "A_s",
        "n*pi*d**2/4",
        n=n,
        d=d
    )


def required_steel_area(M_Ed, z, f_yd):
    """
    Required tensile reinforcement area.
    A_s,req = M_Ed / (z*f_yd)
    """
    return CalcExpression(
        "A_{s,req}",
        "M_Ed/(z*f_yd)",
        M_Ed=M_Ed,
        z=z,
        f_yd=f_yd
    )


def lever_arm(d, x):
    """
    Lever arm.
    z = d - x/3
    """
    return CalcExpression(
        "z",
        "d - x/3",
        d=d,
        x=x
    )


def concrete_area_rectangular(b, h):
    """
    Rectangular concrete area.
    A_c = b*h
    """
    return CalcExpression(
        "A_c",
        "b*h",
        b=b,
        h=h
    )


def second_moment_rectangular(b, h):
    """
    Second moment of area of rectangle about strong axis.
    I = b*h^3/12
    """
    return CalcExpression(
        "I",
        "b*h**3/12",
        b=b,
        h=h
    )


def section_modulus_rectangular(b, h):
    """
    Elastic section modulus of rectangle.
    W = b*h^2/6
    """
    return CalcExpression(
        "W",
        "b*h**2/6",
        b=b,
        h=h
    )


def bending_stress(M, W):
    """
    Bending stress.
    sigma = M/W
    """
    return CalcExpression(
        r"\sigma",
        "M/W",
        M=M,
        W=W
    )


def axial_stress(N, A):
    """
    Axial stress.
    sigma = N/A
    """
    return CalcExpression(
        r"\sigma",
        "N/A",
        N=N,
        A=A
    )


def combined_stress(N, A, M, W):
    """
    Combined axial + bending stress.
    sigma = N/A + M/W
    """
    return CalcExpression(
        r"\sigma",
        "N/A + M/W",
        N=N,
        A=A,
        M=M,
        W=W
    )

In [38]:
As_bar = area_of_bar(d=16)
display(Math(As_bar.report_latex()))

As_total = total_area_of_bars(n=4, d=16)
display(Math(As_total.report_latex()))

I_rect = second_moment_rectangular(b=300, h=500)
display(Math(I_rect.report_latex()))

sigma = combined_stress(
    N=100000,
    A=150000,
    M=50e6,
    W=12.5e6
)
display(Math(sigma.report_latex()))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [40]:
calculations = [
    area_of_bar(d=16),
    total_area_of_bars(n=4, d=16),
    concrete_area_rectangular(b=300, h=500),
    second_moment_rectangular(b=300, h=500),
    section_modulus_rectangular(b=300, h=500),
    combined_stress(N=100000, A=150000, M=50e6, W=12.5e6),
]

for calc in calculations:
    display(Math(calc.report_latex()))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [39]:
A=CalcExpression(
        "A_s",
        "pi*d**2/4",
        d=10
    )
display(Math(A.report_latex()))

<IPython.core.display.Math object>